# Deploy NPS Agent to RHOAI

This notebook deploys the NPS Agent ([`npsagent.py`](./npsagent.py)) to OpenShift AI with MLflow tracing.

## From Local Development to OpenShift Deployment

In the [Evaluate notebook](../1_develop/2_evaluate.ipynb) we built and tested the NPS Agent locally — running it inside a Jupyter cell, calling NPS tools over MCP, and evaluating the results with MLflow scorers.

Now we'll take that **exact same agent logic** and deploy it as a live HTTP endpoint on **Red Hat OpenShift AI (RHOAI)**. The core agent code stays almost identical; we just add a thin serving layer so MLflow can host it as a REST API.

Here's what changes (and what doesn't):

| | Local (evaluate notebook) | Deployed (this notebook) |
|---|---|---|
| **Agent logic** | `run_nps_agent()` with `MCPServerStdio` | Same `run_nps_agent()` with `MCPServerStdio` |
| **MCP server** | Spawned via `uv run fastmcp run` | Same — `uv` is installed in the container |
| **Serving** | Direct `await` in a notebook cell | Wrapped in MLflow `ResponsesAgent` and served over HTTP |
| **Tracing** | `mlflow.openai.autolog()` to local MLflow | Same autolog, but traces go to RHOAI MLflow |
| **Auth** | N/A | RHOAI workspace header for MLflow routing |
| **Infrastructure** | Your laptop | OpenShift pod (s2i build from Git) |


### Cluster Setup

Before continuing, you'll need an OpenShift cluster with RHOAI and MLflow already configured. Follow the [Cluster Setup Guide](https://docs.google.com/document/d/1ZzuGAY1gSamOLsznwbaL7xFkJ1JjHWpnyjk0tV12YVg/edit?tab=t.0#heading=h.jgt5ddlrwyvc) to get your environment ready.

---

## Deployment Files Overview

The `2_deploy/` directory is self-contained with everything OpenShift needs to build and run the agent:

| File | Purpose |
|---|---|
| [`npsagent.py`](./npsagent.py) | The agent — same logic as the evaluate notebook, plus an MLflow `ResponsesAgent` wrapper for HTTP serving |
| [`nps_mcp_server.py`](./nps_mcp_server.py) | FastMCP server exposing NPS API tools (parks, campgrounds, events, etc.) |
| [`app.sh`](./app.sh) | Container entry point — packages the agent with `mlflow.pyfunc.save_model` and starts `mlflow models serve` |
| [`requirements.txt`](./requirements.txt) | Python dependencies installed during the s2i build |
| [`nps-agent.yaml`](./nps-agent.yaml) | OpenShift template defining the BuildConfig, Deployment, Service, and Route |
| [`.s2i/environment`](./.s2i/environment) | Tells s2i to use `app.sh` as the startup script |

The s2i build clones this repo's `deploydemo` branch, scopes to the `2_deploy/` directory, installs `requirements.txt`, and runs `app.sh` on startup.

---

## What Changes in the Agent?

The core `run_nps_agent` function is **identical** between the evaluate notebook and the deployed version. Here it is in the [Evaluate notebook](../1_develop/2_evaluate.ipynb):

```python
# 1_develop/2_evaluate.ipynb
async def run_nps_agent(prompt: str) -> str:
    command = "uv"
    args = ["run", "fastmcp", "run", "./nps_mcp_server.py"]
    env = {"NPS_API_KEY": os.getenv("NPS_API_KEY")}
    async with MCPServerStdio(params={"command": command, "args": args, "env": env}) as mcp_server:
        agent = Agent(
            name="NPS Agent",
            instructions=AGENT_INSTRUCTIONS,
            mcp_servers=[mcp_server],
            model=os.getenv("OPENAI_MODEL_NAME")
        )
        result = await Runner.run(agent, prompt)
        return result.final_output
```

And here it is in [`npsagent.py`](./npsagent.py) — the same function, unchanged:

```python
# 2_deploy/npsagent.py
async def run_nps_agent(prompt: str) -> str:
    command = "uv"
    args = ["run", "fastmcp", "run", "./nps_mcp_server.py"]
    env = {**os.environ, "NPS_API_KEY": os.environ.get("NPS_API_KEY", "")}
    async with MCPServerStdio(params={"command": command, "args": args, "env": env}) as mcp_server:
        agent = Agent(
            name="NPS Agent",
            instructions=AGENT_INSTRUCTIONS,
            mcp_servers=[mcp_server],
            model=os.environ.get("OPENAI_MODEL_NAME", "gpt-4o"),
        )
        result = await Runner.run(agent, prompt)
        return result.final_output
```

The agent logic is the same — `uv` is installed in the container so the MCP server launches exactly as it does locally. The only additions in `npsagent.py` are the **serving wrapper** and **RHOAI tracing header**, described next.

## Wrapping the Agent for HTTP Serving

To serve the agent as an HTTP endpoint, we wrap `run_nps_agent` in an MLflow **`ResponsesAgent`**. This is the class that MLflow's scoring server calls when it receives a `POST /invocations` request:

```python
from mlflow.pyfunc import ResponsesAgent

class NPSResponsesAgent(ResponsesAgent):
    def predict(self, request):
        user_message = self._extract_user_message(request)
        result = asyncio.run(run_nps_agent(user_message))
        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text=result, id="msg_1")]
        )
```

The `predict` method extracts the user's message from the incoming JSON, calls our existing `run_nps_agent`, and returns the result in MLflow's Responses format.

### RHOAI Workspace Header

When tracing to RHOAI's MLflow instance (rather than a local one), we need to attach a workspace header so the Data Science Gateway routes traces to the correct project. This is handled by a small header provider registered at module load time:

```python
_workspace = os.environ.get("MLFLOW_WORKSPACE")
if _workspace:
    class WorkspaceHeader(RequestHeaderProvider):
        def in_context(self):
            return True
        def request_headers(self):
            return {"X-Mlflow-Workspace": os.environ["MLFLOW_WORKSPACE"]}

    _request_header_provider_registry.register(WorkspaceHeader)
```

This only activates when `MLFLOW_WORKSPACE` is set (i.e., on RHOAI). Locally it's a no-op.

### Autologging

Finally, we enable MLflow's OpenAI autologging so every LLM call and tool invocation is automatically traced:

```python
mlflow.openai.autolog()
set_model(NPSResponsesAgent())
```

`set_model` registers our agent class as the model that `app.sh` will package and serve.

---

---

Now that we understand how the agent is adapted for deployment, let's go ahead and deploy it. The steps below will create an OpenShift project, push secrets, build the container, and test the live endpoint.

## Prerequisites

Before you begin, make sure you have:

- An OpenShift cluster with RHOAI and MLflow configured
- The `oc` CLI installed and logged in to your cluster
- An [OpenAI API key](https://platform.openai.com/api-keys)
- An [NPS API key](https://www.nps.gov/subjects/developer/get-started.htm) (free and instant)

In [ ]:
import os
import subprocess

## OpenAI Environment Variables
os.environ["OPENAI_API_KEY"] = ""          # your OpenAI API key
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

## NPS API Key
os.environ["NPS_API_KEY"] = ""             # your NPS API key

# Check that required vars are set
required_vars = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME", "NPS_API_KEY"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set, please set them above.")

## Step 1 — Create an OpenShift Project

Set your namespace below. Use `nps-agent-<yourname>` to avoid conflicts with other users on the same cluster.

In [ ]:
NAMESPACE = ""

In [ ]:
!oc new-project {NAMESPACE}

### Cluster-specific values

Update `MLFLOW_TRACKING_URI` to match your cluster.

In [ ]:
MLFLOW_TRACKING_URI = "https://data-science-gateway.apps.rosa.g4b1z8e6c6k7s6n.2rkm.p3.openshiftapps.com/mlflow/"
MLFLOW_WORKSPACE = NAMESPACE
MLFLOW_EXPERIMENT_NAME = "nps-agent"

## Step 2 — Create the Secret with API Keys

In [ ]:
!oc create secret generic nps-agent-secrets \
  --from-literal=OPENAI_API_KEY="{os.getenv('OPENAI_API_KEY')}" \
  --from-literal=NPS_API_KEY="{os.getenv('NPS_API_KEY')}" \
  -n {NAMESPACE}

## Step 3 — Apply the OpenShift Template

The [`nps-agent.yaml`](./nps-agent.yaml) template creates:
- **BuildConfig** — s2i build from the `deploydemo` branch
- **ImageStream** — stores the built container image
- **Deployment** — runs the agent pod (MLflow serve)
- **Service** — internal cluster networking
- **Route** — external HTTPS endpoint (5 min timeout)

In [ ]:
!oc process -f ./nps-agent.yaml \
  -p NAMESPACE="{NAMESPACE}" \
  -p MLFLOW_TRACKING_URI="{MLFLOW_TRACKING_URI}" \
  -p MLFLOW_WORKSPACE="{MLFLOW_WORKSPACE}" \
  -p MLFLOW_EXPERIMENT_NAME="{MLFLOW_EXPERIMENT_NAME}" \
  | oc apply -f -

---

## Step 4 — Wait for the s2i Build

The BuildConfig triggers automatically. Watch until you see **"Push successful"**.

In [ ]:
!oc logs -f build/nps-agent-1 -n {NAMESPACE}

## Step 5 — Set the MLflow Auth Token

The RHOAI Data Science Gateway requires an auth token. Set it from your current `oc` session.

> **Note:** `oc` tokens expire. Re-run this cell when you need to refresh.

In [ ]:
!oc set env deployment/nps-agent \
  MLFLOW_TRACKING_TOKEN="$(oc whoami -t)" \
  -n {NAMESPACE}

---

## Step 6 — Verify the Pod is Running

In [ ]:
!oc get pods -n {NAMESPACE}

## Step 7 — Get the Route URL

In [ ]:
ROUTE_HOST = subprocess.check_output(
    ["oc", "get", "route", "nps-agent", "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"]
).decode().strip()

AGENT_URL = f"https://{ROUTE_HOST}"
print(f"Agent URL: {AGENT_URL}")

---

## Step 8 — Test the Agent

In [ ]:
import requests
from IPython.display import display, Markdown

payload = {
    "input": [
        {"role": "user", "content": "What national parks are in California?"}
    ]
}

resp = requests.post(
    f"{AGENT_URL}/invocations",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=300,
)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    result = resp.json()
    output_text = result.get("output", [{}])[0].get("text", str(result))
    display(Markdown(output_text))
else:
    print(f"Error: {resp.text}")

---

## Step 9 — View Traces in MLflow

Open your RHOAI MLflow UI and navigate to the **nps-agent** experiment. Every request is auto-traced — you'll see the full chain of LLM calls and MCP tool invocations.

---

## Rebuilding After Code Changes

Push changes to the `deploydemo` branch, then trigger a new build:

In [ ]:
!oc start-build nps-agent -n {NAMESPACE}
!oc logs -f build/nps-agent-2 -n {NAMESPACE}

---

## Cleanup

To delete everything and start from scratch:

In [ ]:
!oc delete project {NAMESPACE}